# Compression-03 — LZ77 : compresser par références dans une fenêtre glissante

**Navigation** : [Index de la série](README.md) · [Compression-02 — Huffman canonique](Compression-02-HuffmanCanonique-Python.ipynb) · [Compression-04 — ANS](Compression-04-ANS-Python.ipynb)

> **Public : Licence.** Prérequis : [Compression-01](Compression-01-ShannonFano-Prefixe-Python.ipynb) (codes préfixes et entropie) et [Compression-02](Compression-02-HuffmanCanonique-Python.ipynb) (longueurs et décodage canonique). Les codes symbole par symbole repaient chaque répétition. LZ77 (Ziv et Lempel, 1977) remplace une suite déjà vue par un couple (décalage, longueur) qui pointe dans la fenêtre des octets déjà produits.
>
> **Durée estimée** : 45 minutes. **Environnement** : .NET 9 ou ultérieur, noyau .NET Interactive.
>
> **Objectifs** : encoder et décoder des jetons LZ77 ; comprendre les copies chevauchantes ; séparer le parse dans `LZData` ; comparer le coût des champs, les estimations d’entropie et le framing ; mesurer une recherche naïve et une recherche guidée par tableau de suffixes.
>
> **Parcours** : corpus réel · fenêtre glissante · trois flux `LZData` · décodage et répétitions · estimations de coût · cadre binaire explicite · paramètres de fenêtre · suffixes et débits. Trois exercices sont répartis dans le carnet.

In [1]:
using System;
using System.Buffers.Binary;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.IO.Compression;
using System.Linq;
using System.Numerics;
using System.Security.Cryptography;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;

record PlotSvg(string Markup);
Formatter.Register(typeof(PlotSvg),
    (obj, writer) => ((TextWriter)writer).Write(((PlotSvg)obj).Markup), "text/html");
Console.WriteLine($"Runtime du noyau : .NET {Environment.Version}; code compatible .NET 9, BCL uniquement.");
Console.WriteLine("Formatter PlotSvg enregistré pour les figures SVG statiques.");

Runtime du noyau : .NET 10.0.14; code compatible .NET 9, BCL uniquement.


Formatter PlotSvg enregistré pour les figures SVG statiques.


## 1. Pourquoi le code par symbole ne suffit plus

Un code par symbole peut s’approcher de l’entropie de l’histogramme, mais cette borne ne modélise pas les dépendances entre octets. Les mots, terminaisons et tournures du français se répètent. LZ77 change donc la représentation : au lieu de recoder une occurrence, il émet une référence vers des octets déjà rencontrés.

Nous reprenons le même texte du *Horla*. Le préfixe de 128 Kio sert à l’empreinte du corpus ; une tranche déterministe de 16 Kio sert aux parsers et aux mesures, comme dans le carnet Python. Les fins de ligne CRLF sont normalisées en LF avant le découpage pour stabiliser le contenu entre plateformes.

In [2]:
static byte[] ReadNormalizedCorpus(string path)
{
    byte[] raw = File.ReadAllBytes(path);
    var normalized = new List<byte>(raw.Length);
    for (int i = 0; i < raw.Length; i++)
    {
        if (raw[i] == (byte)'\r' && i + 1 < raw.Length && raw[i + 1] == (byte)'\n')
            continue;
        normalized.Add(raw[i]);
    }
    return normalized.ToArray();
}

var corpusPath = Path.GetFullPath("../GenAI/Audio/04-Applications/datasets/le_horla_1887.txt");
var corpusBytes = ReadNormalizedCorpus(corpusPath);
var excerptBytes = corpusBytes.Take(131072).ToArray();
var tranche = excerptBytes.Take(16384).ToArray();
var hashCorpus = Convert.ToHexString(SHA256.HashData(excerptBytes)).ToLowerInvariant();
var hashTranche = Convert.ToHexString(SHA256.HashData(tranche)).ToLowerInvariant();
Console.WriteLine($"Corpus : {Path.GetFileName(corpusPath)} | préfixe : {excerptBytes.Length:N0} octets | SHA-256 {hashCorpus}");
Console.WriteLine($"Tranche déterministe : {tranche.Length:N0} octets | SHA-256 {hashTranche}");

var gramPositions = new Dictionary<string, List<int>>();
for (int i = 0; i <= tranche.Length - 16; i++)
{
    string gram = Convert.ToHexString(tranche.AsSpan(i, 16));
    if (!gramPositions.TryGetValue(gram, out var positions))
        gramPositions[gram] = positions = new List<int>();
    positions.Add(i);
}
var mostRepeatedGram = gramPositions.OrderByDescending(item => item.Value.Count).First();
Console.WriteLine($"16-gramme le plus répété : {mostRepeatedGram.Value.Count} occurrences; première distance {mostRepeatedGram.Value[1] - mostRepeatedGram.Value[0]} octets.");

Corpus : le_horla_1887.txt | préfixe : 131,072 octets | SHA-256 248fb4ec30c05351198a40c81969624d447fb8ad36e89d6bb5f8ce6084090704


Tranche déterministe : 16,384 octets | SHA-256 6ad476a15b03231e20eeaa5c75fb97a7bc15118150c1d664cf6bc4175e4e9140


16-gramme le plus répété : 3 occurrences; première distance 42 octets.


### Lecture

Les empreintes et la taille apparaissent dans la sortie, car elles décrivent les données réellement lues. Les 16-grammes qui reviennent à courte distance donnent des candidats à une référence ; la compression dépend ensuite de leur distance, de leur longueur et du coût de ces deux champs.

## 2. Le format de jetons et la recherche naïve

Le parse avance de gauche à droite. À chaque position, la recherche naïve essaie les positions précédentes dans la fenêtre de taille $W$ et garde la correspondance la plus longue, jusqu’à `LONGUEUR_MAX`. Si elle atteint `LONGUEUR_MIN`, l’encodeur émet un couple (offset, longueur) et avance de cette longueur ; sinon, il émet un littéral. La boucle de recherche coûte au pire $O(n·W·L_{max})$ comparaisons d’octets.

### Exemple 1 — Le parse glouton en jetons

L’encodeur naïf complet ci-dessous suit la fenêtre à chaque position, choisit la meilleure correspondance et renvoie les littéraux ou les références. La cellule suivante applique cette implémentation à la tranche réelle.

In [3]:
readonly record struct LzToken(bool IsLiteral, byte Literal, int Offset, int Length)
{
    public static LzToken FromLiteral(byte value) => new(true, value, 0, 0);
    public static LzToken FromMatch(int offset, int length) => new(false, 0, offset, length);
}

const int MinimumMatch = 3;
const int MaximumMatch = 32;
const int DefaultWindow = 1024;

static List<LzToken> EncodeNaive(byte[] data, int window = DefaultWindow,
    int minimumMatch = MinimumMatch, int maximumMatch = MaximumMatch)
{
    if (window <= 0 || minimumMatch <= 0 || maximumMatch < minimumMatch)
        throw new ArgumentOutOfRangeException(nameof(window), "Paramètres de fenêtre ou de longueur invalides.");

    var tokens = new List<LzToken>();
    int position = 0;
    while (position < data.Length)
    {
        int lowerBound = Math.Max(0, position - window);
        int bestLength = 0;
        int bestOffset = 0;
        for (int previous = position - 1; previous >= lowerBound; previous--)
        {
            int length = 0;
            while (length < maximumMatch && position + length < data.Length &&
                   data[previous + length] == data[position + length])
                length++;
            if (length > bestLength)
            {
                bestLength = length;
                bestOffset = position - previous;
                if (length == maximumMatch)
                    break;
            }
        }

        if (bestLength >= minimumMatch)
        {
            tokens.Add(LzToken.FromMatch(bestOffset, bestLength));
            position += bestLength;
        }
        else
        {
            tokens.Add(LzToken.FromLiteral(data[position]));
            position++;
        }
    }
    return tokens;
}

static int LiteralCount(IEnumerable<LzToken> tokens) => tokens.Count(token => token.IsLiteral);
static int MatchCount(IEnumerable<LzToken> tokens) => tokens.Count(token => !token.IsLiteral);

var naiveTokens = EncodeNaive(tranche);
Console.WriteLine($"Parse naïf, fenêtre {DefaultWindow} : {naiveTokens.Count:N0} jetons, {LiteralCount(naiveTokens):N0} littéraux et {MatchCount(naiveTokens):N0} références.");

Parse naïf, fenêtre 1024 : 7,183 jetons, 4,113 littéraux et 3,070 références.


### Lecture

Chaque référence représente plusieurs octets déjà présents dans la sortie. Ce parse reste une représentation intermédiaire : il ne spécifie pas encore comment séparer les jetons du flux de bits, ni comment encoder statistiquement leurs champs.

### Exercice 1 — La taille du décodé, sans décoder

Un parse détermine exactement la taille de sa sortie : chaque littéral produit un octet et chaque référence en produit `Length`. Écrivez `DecodedSize` sans reconstruire les octets; vérifiez le résultat sur le parse complet de la tranche.

In [4]:
static int? DecodedSize(IEnumerable<LzToken> tokens)
{
    // TODO étudiant : sommer les contributions des littéraux et des références.
    // Indice : un littéral contribue 1, une référence contribue son champ Length.
    return null;
}

var decodedSizeByParse = DecodedSize(naiveTokens);
if (decodedSizeByParse is null)
{
    Console.WriteLine("Exercice à compléter");
}
else
{
    if (decodedSizeByParse.Value != tranche.Length)
        throw new Exception("La taille du décodé doit égaler la taille de la tranche.");
    Console.WriteLine($"Exercice 1 : OK — {decodedSizeByParse.Value:N0} octets.");
}

Exercice à compléter


### Lecture

La somme des contributions est égale à la longueur de la tranche. Le parse encode chaque octet exactement une fois comme littéral ou comme partie d’un couple.

## 3. `LZData` : un parse séparé en trois flux

Les codecs séparent les littéraux, les longueurs et les décalages pour appliquer ensuite un code adapté à chacun. `LZData` matérialise ces flux synchronisés. Une longueur et un offset nuls marquent un littéral ; les valeurs non nulles marquent une référence.

In [5]:
sealed record LzData(byte[] Literals, int[] Lengths, int[] Offsets);

static LzData ToLzData(IReadOnlyList<LzToken> tokens)
{
    var literals = new List<byte>();
    var lengths = new int[tokens.Count];
    var offsets = new int[tokens.Count];
    for (int i = 0; i < tokens.Count; i++)
    {
        var token = tokens[i];
        if (token.IsLiteral)
        {
            literals.Add(token.Literal);
            continue;
        }
        lengths[i] = token.Length;
        offsets[i] = token.Offset;
    }
    return new LzData(literals.ToArray(), lengths, offsets);
}

static double Entropy<T>(IEnumerable<T> values) where T : notnull
{
    var counts = new Dictionary<T, int>();
    int total = 0;
    foreach (var value in values)
    {
        counts[value] = counts.GetValueOrDefault(value) + 1;
        total++;
    }
    if (total == 0)
        return 0;
    return -counts.Values.Sum(count =>
    {
        double probability = (double)count / total;
        return probability * Math.Log2(probability);
    });
}

var lzData = ToLzData(naiveTokens);
var matchLengths = naiveTokens.Where(token => !token.IsLiteral).Select(token => token.Length).ToArray();
var matchOffsets = naiveTokens.Where(token => !token.IsLiteral).Select(token => token.Offset).ToArray();
int coveredByReferences = matchLengths.Sum();
Console.WriteLine($"Flux synchronisés : {lzData.Lengths.Length:N0} positions; littéraux {lzData.Literals.Length:N0}; références {matchLengths.Length:N0} couvrant {coveredByReferences:N0} octets.");
Console.WriteLine($"Longueur moyenne d’une référence : {(matchLengths.Length == 0 ? 0 : matchLengths.Average()):F2} octets.");
Console.WriteLine($"Entropies empiriques par symbole : littéraux {Entropy(lzData.Literals):F3} bits/octet; longueurs {Entropy(matchLengths):F3} bits/référence; offsets {Entropy(matchOffsets):F3} bits/référence.");

var lengthHistogram = matchLengths.GroupBy(length => length)
    .OrderBy(group => group.Key).Select(group => (Length: group.Key, Count: group.Count())).ToArray();
int maxHistogramCount = lengthHistogram.Max(item => item.Count);
var histogramSvg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='340' viewBox='0 0 760 340' role='img' aria-label='Histogramme des longueurs LZ77'><rect width='100%' height='100%' fill='white'/><text x='380' y='24' text-anchor='middle' font-family='sans-serif' font-size='17' fill='#222'>Longueurs des références LZ77 — tranche du Horla</text>");
foreach (var item in lengthHistogram)
{
    int x = 54 + (item.Length - MinimumMatch) * 20;
    int barHeight = 235 * item.Count / maxHistogramCount;
    histogramSvg.Append($"<rect x='{x}' y='{270 - barHeight}' width='14' height='{barHeight}' fill='#4C72B0'/>");
    histogramSvg.Append($"<text x='{x + 7}' y='290' text-anchor='middle' font-family='sans-serif' font-size='10' fill='#222'>{item.Length}</text>");
}
histogramSvg.Append("<text x='380' y='322' text-anchor='middle' font-family='sans-serif' font-size='12' fill='#222'>Longueur du couple (octets)</text><text transform='translate(18 160) rotate(-90)' text-anchor='middle' font-family='sans-serif' font-size='12' fill='#222'>Nombre de références</text></svg>");
display(new PlotSvg(histogramSvg.ToString()));
Console.WriteLine($"Histogramme construit pour {lengthHistogram.Length} longueurs observées.");

Flux synchronisés : 7,183 positions; littéraux 4,113; références 3,070 couvrant 12,271 octets.


Longueur moyenne d’une référence : 4.00 octets.


Entropies empiriques par symbole : littéraux 5.080 bits/octet; longueurs 1.956 bits/référence; offsets 9.609 bits/référence.


Longueurs des références LZ77 — tranche du Horla 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 19 21 Longueur du couple (octets) Nombre de références

Histogramme construit pour 17 longueurs observées.


### Lecture

La distribution empirique de la longueur et celle de l’offset peuvent être codées séparément des octets littéraux. Żukowski décrit pour zstd (PivCo-Huffman, section 6.5) que les littéraux deviennent moins dissymétriques après la passe LZ alors que les longueurs de littéraux restent très concentrées. C’est l’intérêt de `LZData` : exposer les trois sources à coder, sans prétendre que leur flux Huffman ou ANS est déjà sérialisé.

## 4. Décoder une copie qui se chevauche

Pour chaque couple $(d, \ell)$, le décodeur ajoute `longueur` octets en lisant `offset` positions derrière la fin courante de la sortie. La longueur peut dépasser l’offset : les octets ajoutés deviennent source des ajouts suivants, ce qui déroule un motif périodique.

In [6]:
static byte[] DecodeTokens(IEnumerable<LzToken> tokens)
{
    var output = new List<byte>();
    foreach (var token in tokens)
    {
        if (token.IsLiteral)
        {
            output.Add(token.Literal);
            continue;
        }
        if (token.Offset <= 0 || token.Offset > output.Count || token.Length <= 0)
            throw new InvalidDataException("Référence en dehors de la sortie déjà produite.");
        for (int i = 0; i < token.Length; i++)
            output.Add(output[output.Count - token.Offset]);
    }
    return output.ToArray();
}

var naiveDecoded = DecodeTokens(naiveTokens);
var naiveHash = Convert.ToHexString(SHA256.HashData(naiveDecoded)).ToLowerInvariant();
if (!naiveDecoded.SequenceEqual(tranche) || naiveHash != hashTranche)
    throw new Exception("Le parse naïf ne restitue pas la tranche exacte.");
Console.WriteLine($"Round-trip tranche : {naiveDecoded.Length:N0} octets; égalité octet à octet et SHA-256 vérifiées ({naiveHash}).");

static void ExpectInvalidReference(string label, LzToken reference)
{
    var tokens = new[] { LzToken.FromLiteral((byte)'x'), reference };
    try
    {
        _ = DecodeTokens(tokens);
    }
    catch (InvalidDataException)
    {
        Console.WriteLine($"Référence invalide rejetée : {label}.");
        return;
    }
    throw new Exception($"La référence invalide aurait dû être rejetée : {label}.");
}

ExpectInvalidReference("offset nul", LzToken.FromMatch(0, 3));
ExpectInvalidReference("offset négatif", LzToken.FromMatch(-1, 3));
ExpectInvalidReference("offset au-delà de la sortie", LzToken.FromMatch(2, 3));

Round-trip tranche : 16,384 octets; égalité octet à octet et SHA-256 vérifiées (6ad476a15b03231e20eeaa5c75fb97a7bc15118150c1d664cf6bc4175e4e9140).


Référence invalide rejetée : offset nul.


Référence invalide rejetée : offset négatif.


Référence invalide rejetée : offset au-delà de la sortie.


### Lecture

Le contrôle compare les octets restaurés et leur empreinte à la tranche d’entrée. La copie chevauchante est une opération de décodage, pas une recherche : elle ne consulte que la sortie déjà produite.

### Exemple 2 — Un run se déplie par référence

Un littéral suivi d’une référence de distance un représente une longue répétition. L’exemple complet ci-dessous rend visible le chevauchement.

In [7]:
var runTokens = new[]
{
    LzToken.FromLiteral((byte)'a'),
    LzToken.FromMatch(offset: 1, length: 15)
};
byte[] runDecoded = DecodeTokens(runTokens);
string runText = Encoding.ASCII.GetString(runDecoded);
if (runText != new string('a', 16))
    throw new Exception("La référence chevauchante n’a pas produit le run attendu.");
Console.WriteLine($"Littéral 'a' + couple (1, 15) : {runText} ({runDecoded.Length} octets).");

Littéral 'a' + couple (1, 15) : aaaaaaaaaaaaaaaa (16 octets).


### Lecture

Le premier octet recopié vient du littéral précédent ; chacun des suivants peut lire un octet que le même couple vient d’ajouter. C’est la raison pour laquelle une longueur supérieure à l’offset est légale.

### Exemple 3 — Vérifier les cas de bord

En plus de la tranche réelle et du run démonstratif, vérifions les alphabets dégénérés, l’alphabet binaire complet et une entrée périodique. Pour chacun, l’encodeur, le décodeur et les empreintes doivent correspondre.

In [8]:
static void CheckEdgeRoundTrip(string label, byte[] input)
{
    var tokens = EncodeNaive(input);
    byte[] decoded = DecodeTokens(tokens);
    string sourceHash = Convert.ToHexString(SHA256.HashData(input)).ToLowerInvariant();
    string decodedHash = Convert.ToHexString(SHA256.HashData(decoded)).ToLowerInvariant();
    if (!input.SequenceEqual(decoded) || sourceHash != decodedHash)
        throw new Exception($"Round-trip de bord échoué : {label}.");
    Console.WriteLine($"{label} : {input.Length} octets, {tokens.Count} jetons, hash vérifié {decodedHash}.");
}

CheckEdgeRoundTrip("Flux vide", Array.Empty<byte>());
CheckEdgeRoundTrip("Octet singleton", new byte[] { 0xA5 });
CheckEdgeRoundTrip("Alphabet binaire 0..255", Enumerable.Range(0, 256).Select(value => (byte)value).ToArray());
CheckEdgeRoundTrip("Motif périodique répété", Encoding.ASCII.GetBytes(string.Concat(Enumerable.Repeat("abab", 32))));

Flux vide : 0 octets, 0 jetons, hash vérifié e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855.


Octet singleton : 1 octets, 1 jetons, hash vérifié 6922e93e3827642ce4b883c756b31abf80036649d3614bf5fcb3adda43b8ea32.


Alphabet binaire 0..255 : 256 octets, 256 jetons, hash vérifié 40aff2e9d2d8922e47afd4648e6967497158785fbd1da870e7110266bf944880.


Motif périodique répété : 128 octets, 6 jetons, hash vérifié 31a4d733c8bc88da3d5d77e5645b7d20c3c4e30825438820c4df72d7eda8e469.


### Exercice 2 — Écrire la primitive de copie

Implémentez `CopyReference(prefix, offset, length)`. Elle doit renvoyer seulement les octets ajoutés, tout en lisant une sortie qui grandit pendant la copie. Testez-la sur `ab` avec offset 2 et longueur 5.

In [9]:
static byte[] CopyReference(byte[] prefix, int offset, int length)
{
    // TODO étudiant : valider offset puis recopier depuis la sortie grandissante.
    // Indice : une liste initialisée avec prefix, puis output.Add(output[^offset]) répété length fois.
    return Array.Empty<byte>();
}

var copied = CopyReference(Encoding.ASCII.GetBytes("ab"), offset: 2, length: 5);
if (copied.Length == 0)
{
    Console.WriteLine("Exercice à compléter");
}
else
{
    byte[] expectedCopy = Encoding.ASCII.GetBytes("ababa");
    if (!copied.SequenceEqual(expectedCopy))
        throw new Exception("La copie doit rendre exactement 'ababa'.");
    Console.WriteLine($"Exercice 2 : OK — {Encoding.ASCII.GetString(copied)}");
}

Exercice à compléter


## 5. Coût des champs : une estimation n’est pas un flux sérialisé

Un littéral paie un bit de type et huit bits de valeur. Une référence paie un bit de type, un champ `offset - 1` de $\lceil\log_2 W\rceil$ bits et un champ `longueur - LONGUEUR_MIN` de largeur suffisante pour les longueurs admises. Ce modèle à champs fixes estime des bits par jeton ; il n’est pas encore un format de fichier.

Nous calculons aussi $nH(X)$ pour l’histogramme des octets. C’est la borne entropique empirique, pas la taille d’un flux Huffman construit. Pour `LZData`, nous additionnons les bornes entropiques des littéraux, longueurs et offsets ; elles ne comprennent ni tables de codes ni framing.

In [10]:
static int BitsRequired(int valueCount)
{
    if (valueCount <= 1)
        return 0;
    return (int)Math.Ceiling(Math.Log2(valueCount));
}

static long FixedFieldBits(IEnumerable<LzToken> tokens, int window,
    int minimumMatch = MinimumMatch, int maximumMatch = MaximumMatch)
{
    int offsetBits = BitsRequired(window);
    int lengthBits = BitsRequired(maximumMatch - minimumMatch + 1);
    long total = 0;
    foreach (var token in tokens)
        total += token.IsLiteral ? 1 + 8 : 1 + offsetBits + lengthBits;
    return total;
}

long rawBits = 8L * tranche.Length;
long fixedTokenBits = FixedFieldBits(naiveTokens, DefaultWindow);
double byteEntropyBound = tranche.Length * Entropy(tranche);
double literalEntropyBound = lzData.Literals.Length * Entropy(lzData.Literals);
double lengthEntropyBound = matchLengths.Length * Entropy(matchLengths);
double offsetEntropyBound = matchOffsets.Length * Entropy(matchOffsets);
double lzEntropyBound = literalEntropyBound + lengthEntropyBound + offsetEntropyBound;
Console.WriteLine($"Tranche : brut {rawBits:N0} bits ({tranche.Length:N0} octets).");
Console.WriteLine($"Estimation champs fixes LZ : {fixedTokenBits:N0} bits; offset {BitsRequired(DefaultWindow)} bits, longueur {BitsRequired(MaximumMatch - MinimumMatch + 1)} bits; framing exclu.");
Console.WriteLine($"Borne empirique n×H(octets) : {byteEntropyBound:N0} bits — borne d’entropie, pas un flux Huffman sérialisé.");
Console.WriteLine($"Somme des bornes H des flux LZData : littéraux {literalEntropyBound:N0} + longueurs {lengthEntropyBound:N0} + offsets {offsetEntropyBound:N0} = {lzEntropyBound:N0} bits; flux de type littéral/référence, codes, tables et framing exclus.");

Tranche : brut 131,072 bits (16,384 octets).


Estimation champs fixes LZ : 86,137 bits; offset 10 bits, longueur 5 bits; framing exclu.


Borne empirique n×H(octets) : 74,881 bits — borne d’entropie, pas un flux Huffman sérialisé.


Somme des bornes H des flux LZData : littéraux 20,892 + longueurs 6,004 + offsets 29,500 = 56,396 bits; flux de type littéral/référence, codes, tables et framing exclus.


### Lecture

Ces sorties décrivent trois grandeurs différentes : un modèle de champs fixes, une borne d’entropie pour les octets bruts, et la somme des bornes d’entropie par flux LZ. Aucune ne prétend être une taille Huffman mesurée. Le champ longueur code `longueur - LONGUEUR_MIN`, donc toutes les longueurs permises tiennent dans la largeur annoncée.

## 6. Sérialiser un cadre pédagogique et compter le framing

Pour rendre concret le coût absent des estimations, voici un petit format pédagogique, défini uniquement pour ce carnet. Son en-tête fixe fait 16 octets : signature, fenêtre, longueur décodée, longueurs minimale et maximale, nombre de jetons. Le payload empaquette un bit de type puis les champs de chaque jeton. Un octet final peut contenir jusqu’à sept bits de bourrage.

Ce n’est ni DEFLATE ni un format de production : les champs des références sont fixes, et les flux `LZData` n’ont pas encore reçu de code d’entropie. Le cadre permet toutefois de distinguer le payload estimé, ses octets sérialisés et le coût explicite du framing.

In [11]:
sealed class BitWriter
{
    private readonly List<byte> data = new();
    public long BitCount { get; private set; }

    public void Write(uint value, int width)
    {
        if (width < 0 || width > 32)
            throw new ArgumentOutOfRangeException(nameof(width));
        for (int shift = width - 1; shift >= 0; shift--)
        {
            if ((BitCount & 7) == 0)
                data.Add(0);
            if (((value >> shift) & 1) != 0)
                data[^1] |= (byte)(1 << (7 - (int)(BitCount & 7)));
            BitCount++;
        }
    }

    public byte[] ToArray() => data.ToArray();
}

sealed class BitReader(byte[] bytes)
{
    private long position;
    public long RemainingBits => (long)bytes.Length * 8 - position;

    public uint Read(int width)
    {
        if (width < 0 || width > 32 || RemainingBits < width)
            throw new InvalidDataException("Payload tronqué.");
        uint value = 0;
        for (int i = 0; i < width; i++)
        {
            value <<= 1;
            value |= (uint)((bytes[(int)(position >> 3)] >> (7 - (int)(position & 7))) & 1);
            position++;
        }
        return value;
    }
}

static byte[] SerializeFrame(IReadOnlyList<LzToken> tokens, int window,
    int originalLength, int minimumMatch = MinimumMatch, int maximumMatch = MaximumMatch)
{
    int offsetBits = BitsRequired(window);
    int lengthBits = BitsRequired(maximumMatch - minimumMatch + 1);
    var writer = new BitWriter();
    foreach (var token in tokens)
    {
        if (token.IsLiteral)
        {
            writer.Write(0, 1);
            writer.Write(token.Literal, 8);
        }
        else
        {
            if (token.Offset < 1 || token.Offset > window ||
                token.Length < minimumMatch || token.Length > maximumMatch)
                throw new InvalidDataException("Jeton hors des limites décrites par l’en-tête.");
            writer.Write(1, 1);
            writer.Write((uint)(token.Offset - 1), offsetBits);
            writer.Write((uint)(token.Length - minimumMatch), lengthBits);
        }
    }

    byte[] payload = writer.ToArray();
    byte[] frame = new byte[16 + payload.Length];
    frame[0] = (byte)'L'; frame[1] = (byte)'Z'; frame[2] = (byte)'7'; frame[3] = (byte)'7';
    BinaryPrimitives.WriteUInt16LittleEndian(frame.AsSpan(4, 2), (ushort)window);
    BinaryPrimitives.WriteUInt32LittleEndian(frame.AsSpan(6, 4), (uint)originalLength);
    frame[10] = (byte)minimumMatch;
    frame[11] = (byte)maximumMatch;
    BinaryPrimitives.WriteUInt32LittleEndian(frame.AsSpan(12, 4), (uint)tokens.Count);
    payload.CopyTo(frame, 16);
    return frame;
}

static (List<LzToken> Tokens, int Window, int OriginalLength) ParseFrame(byte[] frame)
{
    if (frame.Length < 16 || frame[0] != 'L' || frame[1] != 'Z' || frame[2] != '7' || frame[3] != '7')
        throw new InvalidDataException("En-tête LZ77 invalide.");
    int window = BinaryPrimitives.ReadUInt16LittleEndian(frame.AsSpan(4, 2));
    int originalLength = checked((int)BinaryPrimitives.ReadUInt32LittleEndian(frame.AsSpan(6, 4)));
    int minimumMatch = frame[10];
    int maximumMatch = frame[11];
    int tokenCount = checked((int)BinaryPrimitives.ReadUInt32LittleEndian(frame.AsSpan(12, 4)));
    if (window <= 0 || minimumMatch <= 0 || maximumMatch < minimumMatch || tokenCount < 0)
        throw new InvalidDataException("Paramètres d’en-tête invalides.");

    int offsetBits = BitsRequired(window);
    int lengthBits = BitsRequired(maximumMatch - minimumMatch + 1);
    var reader = new BitReader(frame[16..]);
    var tokens = new List<LzToken>(tokenCount);
    for (int i = 0; i < tokenCount; i++)
    {
        if (reader.Read(1) == 0)
            tokens.Add(LzToken.FromLiteral((byte)reader.Read(8)));
        else
        {
            int offset = checked((int)reader.Read(offsetBits)) + 1;
            int length = checked((int)reader.Read(lengthBits)) + minimumMatch;
            if (offset > window || length > maximumMatch)
                throw new InvalidDataException("Référence hors des limites déclarées par l’en-tête.");
            tokens.Add(LzToken.FromMatch(offset, length));
        }
    }
    return (tokens, window, originalLength);
}

static byte[] DecodeFrame(byte[] frame)
{
    var parsed = ParseFrame(frame);
    byte[] decoded = DecodeTokens(parsed.Tokens);
    if (decoded.Length != parsed.OriginalLength)
        throw new InvalidDataException("La taille décodée ne correspond pas à la longueur déclarée.");
    return decoded;
}

Console.WriteLine("Sérialiseur et lecteur du cadre LZ77 définis : en-tête validé, 16 octets, payload MSB-first.");

Sérialiseur et lecteur du cadre LZ77 définis : en-tête validé, 16 octets, payload MSB-first.


Un exemple complet chiffre séparément les champs, leur padding et l’en-tête. Le parsing du cadre suivi du décodeur vérifie que la représentation sérialisée restitue la même tranche.

In [12]:
var frameBytes = SerializeFrame(naiveTokens, DefaultWindow, tranche.Length);
var parsedFrame = ParseFrame(frameBytes);
var framedDecoded = DecodeFrame(frameBytes);
int fixedPayloadBytes = (int)((fixedTokenBits + 7) / 8);
int framedBytesExpected = 16 + fixedPayloadBytes;
if (!framedDecoded.SequenceEqual(tranche) || parsedFrame.Window != DefaultWindow ||
    parsedFrame.OriginalLength != tranche.Length || frameBytes.Length != framedBytesExpected)
    throw new Exception("Le cadre sérialisé ne respecte pas son framing ou son round-trip.");
var emptyFrame = SerializeFrame(Array.Empty<LzToken>(), DefaultWindow, 0);
var emptyFrameParsed = ParseFrame(emptyFrame);
if (emptyFrameParsed.OriginalLength != 0 || emptyFrameParsed.Tokens.Count != 0 ||
    DecodeTokens(emptyFrameParsed.Tokens).Length != 0)
    throw new Exception("Le cadre vide doit décoder vers un flux vide.");
int payloadPaddingBits = (int)(fixedPayloadBytes * 8 - fixedTokenBits);
Console.WriteLine($"Champs estimés : {fixedTokenBits:N0} bits; payload sérialisé : {fixedPayloadBytes:N0} octets, dont {payloadPaddingBits} bits de padding.");
Console.WriteLine($"Framing défini : 16 octets; cadre entier : {frameBytes.Length:N0} octets; round-trip cadre : exact.");
Console.WriteLine($"Cadre vide : {emptyFrame.Length} octets de framing et aucun payload; décodage vide exact.");
Console.WriteLine($"Référence réelle .NET ZLibStream reste mesurée plus loin; elle inclut son propre framing et son codage entropique.");

var boundaryTokens = new[] { LzToken.FromLiteral((byte)'a'), LzToken.FromMatch(1, 3) };
byte[] validSmallWindowFrame = SerializeFrame(boundaryTokens, window: 3, originalLength: 4);
var badOffsetFrame = (byte[])validSmallWindowFrame.Clone();
badOffsetFrame[17] |= 0b0011_0000; // code offset 3 => offset 4, supérieur à W=3
var badLengthFrame = (byte[])validSmallWindowFrame.Clone();
badLengthFrame[17] |= 0b0000_1111;
badLengthFrame[18] |= 0b1000_0000; // code 31 => longueur 34, supérieure à Lmax=32
var badDeclaredSizeFrame = SerializeFrame(boundaryTokens, window: 3, originalLength: 99);
static void ExpectInvalidFrame(string label, byte[] frame)
{
    try
    {
        _ = DecodeFrame(frame);
    }
    catch (InvalidDataException)
    {
        Console.WriteLine($"Trame invalide rejetée : {label}.");
        return;
    }
    throw new Exception($"La trame invalide aurait dû être rejetée : {label}.");
}
ExpectInvalidFrame("offset supérieur à la fenêtre", badOffsetFrame);
ExpectInvalidFrame("longueur supérieure à Lmax", badLengthFrame);
ExpectInvalidFrame("longueur décodée différente de l’en-tête", badDeclaredSizeFrame);

Champs estimés : 86,137 bits; payload sérialisé : 10,768 octets, dont 7 bits de padding.


Framing défini : 16 octets; cadre entier : 10,784 octets; round-trip cadre : exact.


Cadre vide : 16 octets de framing et aucun payload; décodage vide exact.


Référence réelle .NET ZLibStream reste mesurée plus loin; elle inclut son propre framing et son codage entropique.


Trame invalide rejetée : offset supérieur à la fenêtre.


Trame invalide rejetée : longueur supérieure à Lmax.


Trame invalide rejetée : longueur décodée différente de l’en-tête.


### Lecture

Pour ce cadre précis, le coût sur disque est exactement l’en-tête choisi plus les octets du payload, avec le padding nécessaire au dernier octet. Les 16 octets de framing ne décrivent pas le format zlib. Une production remplacerait également nos champs fixes et brancherait un code d’entropie réel sur les flux.

### Exercice 3 — Choisir une fenêtre selon le coût des champs

Pour chaque fenêtre candidate, le parse glouton peut trouver d’autres références et la largeur du champ offset change. Implémentez `BestWindow`, puis vérifiez que sa réponse correspond au minimum recalculé indépendamment sur un banc court.

In [13]:
static (int Window, long Bits)? BestWindow(byte[] data, int[] candidates)
{
    // TODO étudiant : parser avec chaque fenêtre puis comparer FixedFieldBits.
    return null;
}

byte[] windowBench = tranche.Take(8192).ToArray();
int[] windowCandidates = { 128, 256, 512, 1024 };
var bestWindow = BestWindow(windowBench, windowCandidates);
if (bestWindow is null)
{
    Console.WriteLine("Exercice à compléter");
}
else
{
    var measuredCosts = windowCandidates.Select(window =>
        (Window: window, Bits: FixedFieldBits(EncodeNaive(windowBench, window), window))).ToArray();
    var expectedBest = measuredCosts.OrderBy(item => item.Bits).ThenBy(item => item.Window).First();
    if (bestWindow.Value != expectedBest)
        throw new Exception("La fenêtre retournée n’est pas le minimum mesuré.");
    Console.WriteLine($"Exercice 3 : OK — fenêtre {bestWindow.Value.Window}, estimation {bestWindow.Value.Bits:N0} bits.");
    Console.WriteLine("Courbe : " + string.Join(" | ", measuredCosts.Select(item => $"W={item.Window}: {item.Bits:N0} bits")));
}

Exercice à compléter


### Exemple 4 — Balayer plusieurs tailles de fenêtre

Ce relevé complet utilise la tranche de 16 Kio. L’exercice précédent travaille sur un préfixe plus court et demande d’implémenter la sélection du meilleur budget.

In [14]:
int[] windowGridValues = { 128, 256, 512, 1024 };
var windowGrid = windowGridValues.Select(window =>
{
    var tokens = EncodeNaive(tranche, window);
    long bitEstimate = FixedFieldBits(tokens, window);
    int framedSize = SerializeFrame(tokens, window, tranche.Length).Length;
    return (Window: window, Tokens: tokens.Count, Matches: MatchCount(tokens),
        Bits: bitEstimate, FrameBytes: framedSize);
}).ToArray();
foreach (var row in windowGrid)
    Console.WriteLine($"W={row.Window,4} : {row.Tokens,6:N0} jetons, {row.Matches,6:N0} références, {row.Bits,8:N0} bits estimés, {row.FrameBytes,6:N0} octets de cadre.");

int plotMaxBits = (int)windowGrid.Max(row => row.Bits);
var gridSvg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='640' height='300' viewBox='0 0 640 300' role='img' aria-label='Coût estimé selon la taille de fenêtre'><rect width='100%' height='100%' fill='white'/><text x='320' y='24' text-anchor='middle' font-family='sans-serif' font-size='16' fill='#222'>Estimation des champs LZ77 selon la fenêtre</text>");
for (int i = 0; i < windowGrid.Length; i++)
{
    int x = 120 + i * 120;
    int barHeight = 190 * (int)windowGrid[i].Bits / plotMaxBits;
    gridSvg.Append($"<rect x='{x}' y='{225 - barHeight}' width='56' height='{barHeight}' fill='#4C72B0'/>");
    gridSvg.Append($"<text x='{x + 28}' y='246' text-anchor='middle' font-family='sans-serif' font-size='12' fill='#222'>W={windowGrid[i].Window}</text>");
    gridSvg.Append($"<text x='{x + 28}' y='{220 - barHeight}' text-anchor='middle' font-family='sans-serif' font-size='10' fill='#222'>{windowGrid[i].Bits:N0}</text>");
}
gridSvg.Append("<text x='320' y='280' text-anchor='middle' font-family='sans-serif' font-size='12' fill='#222'>Bits estimés avec des champs fixes; framing exclu</text></svg>");
display(new PlotSvg(gridSvg.ToString()));
Console.WriteLine($"Grille calculée sur {tranche.Length:N0} octets; la figure compare les bits estimés, sans les présenter comme un flux écrit.");

W= 128 : 12,074 jetons,  1,487 références,  114,614 bits estimés, 14,343 octets de cadre.


W= 256 : 10,361 jetons,  2,105 références,  103,774 bits estimés, 12,988 octets de cadre.


W= 512 :  8,706 jetons,  2,652 références,   94,266 bits estimés, 11,800 octets de cadre.


W=1024 :  7,183 jetons,  3,070 références,   86,137 bits estimés, 10,784 octets de cadre.


Estimation des champs LZ77 selon la fenêtre W=128 114,614 W=256 103,774 W=512 94,266 W=1024 86,137 Bits estimés avec des champs fixes; framing exclu

Grille calculée sur 16,384 octets; la figure compare les bits estimés, sans les présenter comme un flux écrit.


## 7. La recherche : du scan exhaustif au tableau de suffixes

Le parser naïf rescane la fenêtre à chaque position. Le constructeur par doublement trie les rangs des suffixes à chaque puissance de deux : $O(n log^2 n)$ pour cette version simple. Une recherche binaire localise ensuite le voisinage des meilleurs candidats. La version pédagogique ci-dessous borne les sondes autour du point d’insertion : elle peut choisir un couple plus court que la recherche exhaustive. Le round-trip vérifie la validité, tandis que le coût estimé mesure la qualité du parse.

In [15]:
static int[] BuildSuffixArray(byte[] data)
{
    int n = data.Length;
    var suffixes = Enumerable.Range(0, n).ToArray();
    var ranks = data.Select(value => (int)value).ToArray();
    var nextRanks = new int[n];
    for (int width = 1; width < n; width *= 2)
    {
        Array.Sort(suffixes, (left, right) =>
        {
            int first = ranks[left].CompareTo(ranks[right]);
            if (first != 0)
                return first;
            int leftSecond = left + width < n ? ranks[left + width] : -1;
            int rightSecond = right + width < n ? ranks[right + width] : -1;
            return leftSecond.CompareTo(rightSecond);
        });

        if (n == 0)
            break;
        nextRanks[suffixes[0]] = 0;
        for (int i = 1; i < n; i++)
        {
            int previous = suffixes[i - 1];
            int current = suffixes[i];
            bool different = ranks[previous] != ranks[current] ||
                (previous + width < n ? ranks[previous + width] : -1) !=
                (current + width < n ? ranks[current + width] : -1);
            nextRanks[current] = nextRanks[previous] + (different ? 1 : 0);
        }
        (ranks, nextRanks) = (nextRanks, ranks);
        if (ranks[suffixes[^1]] == n - 1)
            break;
    }
    return suffixes;
}

static int ComparePrefixes(byte[] data, int suffixPosition, int queryPosition, int maximumLength)
{
    int left = suffixPosition;
    int right = queryPosition;
    for (int compared = 0; compared < maximumLength; compared++)
    {
        bool leftEnded = left >= data.Length;
        bool rightEnded = right >= data.Length;
        if (leftEnded || rightEnded)
            return leftEnded == rightEnded ? 0 : leftEnded ? -1 : 1;
        int comparison = data[left].CompareTo(data[right]);
        if (comparison != 0)
            return comparison;
        left++;
        right++;
    }
    return 0;
}

static List<LzToken> EncodeWithSuffixArray(byte[] data, int[] suffixArray,
    int window = DefaultWindow, int neighborhood = 32)
{
    var tokens = new List<LzToken>();
    int position = 0;
    while (position < data.Length)
    {
        int low = 0;
        int high = suffixArray.Length;
        while (low < high)
        {
            int middle = (low + high) / 2;
            if (ComparePrefixes(data, suffixArray[middle], position, MaximumMatch) < 0)
                low = middle + 1;
            else
                high = middle;
        }

        int lowerBound = Math.Max(0, position - window);
        int bestLength = 0;
        int bestOffset = 0;
        int scanStart = Math.Max(0, low - neighborhood);
        int scanEnd = Math.Min(suffixArray.Length, low + neighborhood);
        for (int rank = scanStart; rank < scanEnd; rank++)
        {
            int previous = suffixArray[rank];
            if (previous < lowerBound || previous >= position)
                continue;
            int length = 0;
            while (length < MaximumMatch && position + length < data.Length &&
                   data[previous + length] == data[position + length])
                length++;
            if (length > bestLength)
            {
                bestLength = length;
                bestOffset = position - previous;
            }
        }

        if (bestLength >= MinimumMatch)
        {
            tokens.Add(LzToken.FromMatch(bestOffset, bestLength));
            position += bestLength;
        }
        else
        {
            tokens.Add(LzToken.FromLiteral(data[position]));
            position++;
        }
    }
    return tokens;
}

Console.WriteLine("Constructeur de suffixes et recherche guidée définis; sondage d’au plus 64 candidats (32 de chaque côté du point d’insertion).");

Constructeur de suffixes et recherche guidée définis; sondage d’au plus 64 candidats (32 de chaque côté du point d’insertion).


La construction du tableau et le parse guidé sont chronométrés séparément. Le zlib de .NET sert de référence réelle et sa sortie est décompressée pour valider la mesure. Les vitesses dépendent de la machine; les tailles et débits affichés restent dans les sorties, pas dans la prose.

In [16]:
static byte[] CompressZlib(byte[] data)
{
    using var output = new MemoryStream();
    using (var compressor = new ZLibStream(output, CompressionLevel.SmallestSize, leaveOpen: true))
        compressor.Write(data);
    return output.ToArray();
}

static byte[] DecompressZlib(byte[] data)
{
    using var input = new MemoryStream(data);
    using var decompressor = new ZLibStream(input, CompressionMode.Decompress);
    using var output = new MemoryStream();
    decompressor.CopyTo(output);
    return output.ToArray();
}

const int BenchmarkRepetitions = 5;
var buildTimer = Stopwatch.StartNew();
int[] suffixArray = BuildSuffixArray(tranche);
buildTimer.Stop();
var naiveDurations = new List<double>();
var suffixDurations = new List<double>();
var coldSuffixDurations = new List<double>();
var zlibDurations = new List<double>();
for (int repetition = 0; repetition < BenchmarkRepetitions; repetition++)
{
    var timer = Stopwatch.StartNew();
    _ = EncodeNaive(tranche);
    timer.Stop();
    naiveDurations.Add(timer.Elapsed.TotalSeconds);

    timer.Restart();
    _ = EncodeWithSuffixArray(tranche, suffixArray);
    timer.Stop();
    suffixDurations.Add(timer.Elapsed.TotalSeconds);

    timer.Restart();
    _ = EncodeWithSuffixArray(tranche, BuildSuffixArray(tranche));
    timer.Stop();
    coldSuffixDurations.Add(timer.Elapsed.TotalSeconds);

    timer.Restart();
    _ = CompressZlib(tranche);
    timer.Stop();
    zlibDurations.Add(timer.Elapsed.TotalSeconds);
}

static double MedianSeconds(IEnumerable<double> samples)
{
    var ordered = samples.OrderBy(value => value).ToArray();
    return ordered[ordered.Length / 2];
}

var suffixTokens = EncodeWithSuffixArray(tranche, suffixArray);
var suffixDecoded = DecodeTokens(suffixTokens);
var zlibBytes = CompressZlib(tranche);
var zlibDecoded = DecompressZlib(zlibBytes);
if (!suffixDecoded.SequenceEqual(tranche) || !zlibDecoded.SequenceEqual(tranche))
    throw new Exception("Le parser suffixe ou le témoin zlib n’a pas restitué la tranche.");

double trancheMib = (double)tranche.Length / (1024 * 1024);
Console.WriteLine($"Benchmark : {tranche.Length:N0} octets, médiane de {BenchmarkRepetitions} répétitions; runtime .NET {Environment.Version}.");
Console.WriteLine($"Parse naïf : {trancheMib / MedianSeconds(naiveDurations):F2} Mio/s.");
Console.WriteLine($"Table de suffixes, construction incluse : {trancheMib / MedianSeconds(coldSuffixDurations):F2} Mio/s; construction mesurée séparément {buildTimer.Elapsed.TotalSeconds:F3} s.");
Console.WriteLine($"Table de suffixes préconstruite : {trancheMib / MedianSeconds(suffixDurations):F2} Mio/s.");
Console.WriteLine($"ZLibStream .NET (SmallestSize) : {trancheMib / MedianSeconds(zlibDurations):F2} Mio/s; taille comprimée réellement mesurée {zlibBytes.Length:N0} octets.");
Console.WriteLine($"Ratios au brut : n×H(octets) {(100 * byteEntropyBound / rawBits):F1} % et somme des bornes LZData {(100 * lzEntropyBound / rawBits):F1} % (estimations); ZLibStream {(100.0 * zlibBytes.Length / tranche.Length):F1} % (flux réel avec framing).");
Console.WriteLine($"Estimation champs fixes : naïf {FixedFieldBits(naiveTokens, DefaultWindow):N0} bits; suffixes {FixedFieldBits(suffixTokens, DefaultWindow):N0} bits (deux parses valides, qualité possiblement différente).");

Benchmark : 16,384 octets, médiane de 5 répétitions; runtime .NET 10.0.14.


Parse naïf : 0.50 Mio/s.


Table de suffixes, construction incluse : 0.73 Mio/s; construction mesurée séparément 0.015 s.


Table de suffixes préconstruite : 1.71 Mio/s.


ZLibStream .NET (SmallestSize) : 20.78 Mio/s; taille comprimée réellement mesurée 7,376 octets.


Ratios au brut : n×H(octets) 57.1 % et somme des bornes LZData 43.0 % (estimations); ZLibStream 45.0 % (flux réel avec framing).


Estimation champs fixes : naïf 86,137 bits; suffixes 86,368 bits (deux parses valides, qualité possiblement différente).


### Lecture

Le tableau de suffixes échange un coût de construction contre une recherche plus ciblée ; la mesure distingue le parse avec tableau préconstruit et le parcours qui inclut sa construction. Les sondes bornées peuvent manquer un meilleur couple, donc les deux parses ne sont pas tenus d’avoir la même taille. Le round-trip certifie leur validité. La sortie place aussi côte à côte $nH(X)$, la somme des bornes par flux et le ratio de la sortie `ZLibStream`. Les deux premières sont des estimations entropiques sans tables de codes ni framing; la dernière est un flux réellement comprimé avec son propre framing. `ZLibStream` est une implémentation BCL réellement exécutée, pas notre parse LZ77.

## À retenir

| Notion | Ce qu’elle dit | Où la voir |
|---|---|---|
| Parse LZ77 | Remplacer une occurrence répétée par un couple offset/longueur | Sections 2–3 |
| Copie chevauchante | La sortie grandissante permet longueur > offset | Section 4 |
| `LZData` | Littéraux, longueurs et offsets forment trois flux distincts | Section 3 |
| Estimation d’entropie | $nH$ est une borne empirique, pas un flux Huffman sérialisé | Section 5 |
| Framing | Le cadre de démonstration compte séparément en-tête, payload et padding | Section 6 |
| Recherche | Tableau de suffixes et zlib ont des coûts et contrats mesurés distincts | Section 7 |

LZ77 exploite les répétitions en remplaçant des octets par des références dans la sortie passée. La copie chevauchante couvre naturellement les runs. Le parse est distinct du codage des trois flux, et les estimations de Shannon ne sont pas des tailles de flux codés. Le cadre pédagogique explicite son en-tête et son padding; zlib sert de mesure de production via `ZLibStream`.

Références : J. Ziv et A. Lempel, « A universal algorithm for sequential data compression », *IEEE Transactions on Information Theory* 23(3), 1977 · DEFLATE, RFC 1951 · M. Żukowski, *PivCo-Huffman* v1.0, section 6.5.

## Vérifiez votre compréhension

1. Pourquoi une longueur supérieure à l’offset peut-elle être décodée ?

<details><summary>Réponse</summary>

Parce que chaque octet ajouté devient une source possible pour les prochaines lectures du même couple.

</details>

2. Que représente $nH(X)$ dans ce carnet ?

<details><summary>Réponse</summary>

Une borne entropique calculée depuis l’histogramme, pas un flux Huffman construit ni sérialisé.

</details>

3. Quels coûts distingue le petit cadre binaire ?

<details><summary>Réponse</summary>

Le payload des champs, le padding pour terminer l’octet et l’en-tête de framing. La table de suffixes est un coût de calcul, pas une partie du fichier.

</details>

## Pour aller plus loin

- [Compression-04 — ANS](Compression-04-ANS-Python.ipynb) applique un code d’entropie aux distributions de symboles.
- [Compression-04b — codecs SOTA](Compression-04b-CodecsSOTA-Pareto-Python.ipynb) compare des implémentations de production, dont zstd et Brotli.
- DEFLATE (RFC 1951) assemble une passe LZ77 et un codage Huffman dans un format publié.

Le fil complet de la série : [EPIC #18706](https://github.com/jsboige/CoursIA/issues/18706).